# Prototype: llm_judge

Reproduction check for `detectors/llm_judge.py`. Not run by the pipeline: upload to Colab (File > Upload notebook).

**Runtime:** CPU runtime, Colab secret OPENROUTER_API_KEY. Set JUDGE in cell 0 first (about $0.05 to $2 per run, see the judge table).

**Decides:** Rerun Djire's zero-shot judge on his 144 samples and compare with Table 1 in both directions (primary check: kappa with his shipped verdicts at or above his own between-run agreement).

Run top to bottom, then paste the printed comparison back into the chat.

**Cost:** for a judge without reasoning tokens, set `VARIANTS = {"paper": 1024}` to halve the calls.

In [ ]:
# %% 0. Config. Set JUDGE yourself (see the judge list); nothing else needs editing.
# %pip install -q "openai>=1.0.0" "scikit-learn>=1.3.0" "pandas>=2.0.0"   # Colab already has these
import os, json, re, zipfile, urllib.request, concurrent.futures as cf
from collections import Counter
import numpy as np
from openai import OpenAI
from sklearn.metrics import cohen_kappa_score, roc_auc_score

JUDGE = ""                      # the OpenRouter ID you call (see the judge list)
PAPER_JUDGE = ""                # the Table 1 judge it stands in for; "" = same as JUDGE
STRATEGY = "classification"     # Djire's direct zero-shot prompt (Table 1 "Zero-shot")
ITERS, TEMPERATURE = 3, 0.1     # Djire §3
VARIANTS = {"paper": 1024, "fixed": 8192}   # max_tokens: paper's llm4gcd.py call_model uses 1024
RETRIES_FIXED = 3               # "fixed" re-asks on an unparseable reply; "paper" does not
WORKERS = 8
assert JUDGE, "set JUDGE"
PAPER_JUDGE = PAPER_JUDGE or JUDGE
try:
    from google.colab import userdata; os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY"))
except Exception:
    pass
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

In [ ]:
# %% 1. Fetch the replication package (20.9 MB): dataset + the paper's Table 1 confusion matrices + raw verdicts
ROOT = "/content/llm4gcd"
if not os.path.exists(f"{ROOT}/llm4gcd.py"):
    urllib.request.urlretrieve("https://zenodo.org/api/records/22248135/files/src.zip/content", "/content/src.zip")
    zipfile.ZipFile("/content/src.zip").extractall(ROOT)
data = json.load(open(f"{ROOT}/data/AI-Human-Generated-Program-Code-Dataset.jsonl"))  # a JSON array, not JSONL
# build_sample_list(): 18 human (from the ChatGPT rows), 18 ChatGPT, then the 108 other AI rows -> idx 0..143
chat = [d for d in data if d["generator_name"] == "ChatGPT"]
other = [d for d in data if d["generator_name"] != "ChatGPT"]
S = [dict(human=True, gen=None, lang=d["language"], code=d["human_generated_code"]) for d in chat] \
  + [dict(human=False, gen=d["generator_name"], lang=d["language"], code=d["ai_generated_code"]) for d in chat + other]
assert len(S) == 144 and sum(s["human"] for s in S) == 18
GENS = ["StarCoder", "ChatGPT", "Bing AI Chat", "Code Llama", "CodeWhisperer", "GitHub Copilot", "InstructCodeT5+"]
TABLE1 = json.load(open(f"{ROOT}/output/results_new_run.json"))["results"]   # every Table 1 cell is this file's cm

In [ ]:
# %% 2. Djire's exact prompts (llm4gcd.py PROMPTS), sent as the SYSTEM message; the bare code is the USER message
PROMPTS = {
"classification": """
Analyze the provided code snippet to determine whether it was generated by a Large Language Model or written by a human. Prepare a JSON response with two key attributes:
1. "response":
   - Use "yes" if the code appears to be LLM-generated
   - Use "no" if the code seems to be human-written
2. "explanation":
   - Provide a justification of your answer
""",
"classification_stylo_semantic": """
Examine the given code snippet. Begin by determining its purpose and identifying any stylometric and semantic characteristics.
Then, based on these observation, determine whether the code was generated by a LLM or written by a human. Then, prepare a JSON response with the following two key attributes:
1. "response":
   - Use "yes" if the code appears to be LLM-generated
   - Use "no" if the code seems to be human-written
2. "explanation":
   - Provide a justification of your answer
""",
}

def parse(text):  # llm4gcd.py parse_response, condensed, plus a regex fallback; returns "yes" | "no" | None
    text = (text or "").strip()
    for cand in [text, *re.findall(r'\{[^{}]*"response"[^{}]*\}', text, re.S),
                 text[text.find("{"):text.rfind("}") + 1], *re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.S)]:
        try:
            v = str(json.loads(cand).get("response", "")).strip().lower()
            return v if v in ("yes", "no") else None
        except Exception:
            pass
    # Fallback: Mistral models put raw newlines inside the JSON string, which json.loads rejects. Table 1 is
    # reproduced only when these replies are recovered (results_new_run.json counts them as n_recovered).
    m = re.search(r'"response"\s*:\s*"\s*(yes|no)\b', text, re.I)
    return m.group(1).lower() if m else None

eps = json.load(urllib.request.urlopen(f"https://openrouter.ai/api/v1/models/{JUDGE}/endpoints"))["data"]["endpoints"]
if not any("temperature" in e.get("supported_parameters", []) for e in eps):
    print(f"WARNING: no {JUDGE} endpoint accepts temperature; T={TEMPERATURE} is not applied (true for the paper's run too)")

In [ ]:
# %% 3. Inference, cached per (variant, iter, idx) so a rerun resumes
CACHE = f"/content/judge_{JUDGE.replace('/', '_')}_{STRATEGY}.json"
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def ask(code, max_tokens):
    try:
        r = client.chat.completions.create(model=JUDGE, temperature=TEMPERATURE, max_tokens=max_tokens,
            messages=[{"role": "system", "content": PROMPTS[STRATEGY]}, {"role": "user", "content": code}])
        u = r.usage; det = getattr(u, "completion_tokens_details", None)
        return dict(v=parse(r.choices[0].message.content), finish=r.choices[0].finish_reason,
                    out=getattr(u, "completion_tokens", None), reas=getattr(det, "reasoning_tokens", None) if det else None)
    except Exception as e:
        return dict(v=None, finish=f"exception: {e}"[:200], out=None, reas=None)

def job(key):
    variant, it, idx = key.split("|")
    for _ in range(1 + (RETRIES_FIXED if variant == "fixed" else 0)):
        rec = ask(S[int(idx)]["code"], VARIANTS[variant])
        if rec["v"]:
            break
    return key, rec

todo = [f"{v}|{it}|{i}" for v in VARIANTS for it in range(1, ITERS + 1) for i in range(144) if f"{v}|{it}|{i}" not in cache]
with cf.ThreadPoolExecutor(WORKERS) as ex:
    for n, (k, rec) in enumerate(ex.map(job, todo), 1):
        cache[k] = rec
        if n % 50 == 0:
            json.dump(cache, open(CACHE, "w")); print(n, "/", len(todo))
json.dump(cache, open(CACHE, "w"))

In [ ]:
# %% 4. Score = share of "yes" (AI) among valid verdicts; verdict = Score > 0.5 (majority). Report both directions.
def scores(variant):
    out = []
    for i in range(144):
        v = [cache[f"{variant}|{it}|{i}"]["v"] for it in range(1, ITERS + 1)]
        v = [a for a in v if a]
        out.append(sum(a == "yes" for a in v) / len(v) if v else None)
    return out

def paper_raw_verdicts():  # the paper's shipped rs_*.json, majority of valid (recovered) votes
    out = []
    for i in range(144):
        rs = [json.load(open(f"{ROOT}/output/{STRATEGY}/{PAPER_JUDGE}/iter_{it}/rs_{i}.json")) for it in (1, 2, 3)]
        v = [r["response"] if r["response"] in ("yes", "no") else parse(r["explanation"]) for r in rs]  # error text keeps 300 raw chars
        v = [a for a in v if a]
        out.append(None if not v else sum(a == "yes" for a in v) / len(v) > 0.5)
    return out

def f1s(cm):  # cm in the paper's human-positive layout
    tp, fp, fn, tn = cm["tp"], cm["fp"], cm["fn"], cm["tn"]
    return (2*tp/(2*tp+fp+fn) if tp else 0.0), (2*tn/(2*tn+fn+fp) if tn else 0.0), (fp/(tn+fp) if tn+fp else np.nan)

for variant in VARIANTS:
    recs = [cache[f"{variant}|{it}|{i}"] for it in range(1, ITERS + 1) for i in range(144)]
    print(f"\n=== {JUDGE} | {STRATEGY} | variant={variant} (max_tokens={VARIANTS[variant]}) ===")
    print("finish_reason:", Counter(r["finish"] for r in recs).most_common(4),
          "| invalid replies:", sum(r["v"] is None for r in recs), "/", len(recs),
          "| mean reasoning tokens:", np.mean([r["reas"] for r in recs if r["reas"] is not None] or [np.nan]))
    sc = scores(variant)
    print("samples with no valid vote: human", sum(s["human"] and x is None for s, x in zip(S, sc)),
          "/ 18, AI", sum((not s["human"]) and x is None for s, x in zip(S, sc)), "/ 126")
    print(f"{'generator':16s} {'n':>3} {'err':>3} | ours hF1  T1 hF1 | ours llmF1  T1 llmF1 | ours miss  T1 miss")
    ours_all, t1_all = [], []
    for g in GENS:
        rows = [(s, x) for s, x in zip(S, sc) if s["human"] or s["gen"] == g]
        ev = [(s, x) for s, x in rows if x is not None]                       # paper-style: drop invalid
        cm = dict(tp=sum(s["human"] and x <= .5 for s, x in ev), fn=sum(s["human"] and x > .5 for s, x in ev),
                  tn=sum((not s["human"]) and x > .5 for s, x in ev), fp=sum((not s["human"]) and x <= .5 for s, x in ev))
        o, t = f1s(cm), f1s(TABLE1[STRATEGY][PAPER_JUDGE]["per_ai_model"][g]["cm"])
        ours_all.append(o); t1_all.append(t)
        print(f"{g:16s} {len(ev):3d} {len(rows)-len(ev):3d} | {o[0]:8.2f} {t[0]:7.2f} | {o[1]:10.2f} {t[1]:9.2f} | {o[2]:9.2f} {t[2]:8.2f}")
    o, t = np.nanmean(ours_all, 0), np.nanmean(t1_all, 0)
    print(f"{'MEAN':16s}         | {o[0]:8.2f} {t[0]:7.2f} | {o[1]:10.2f} {t[1]:9.2f} | {o[2]:9.2f} {t[2]:8.2f}")
    # Secondary check. +-0.05 on mean F1 is a judgment call (inference): the shipped verdicts rebuild Table 1 to within
    # 0.03 mean / 0.08 worst cell, but that measures parse recovery, not sampling noise.
    print("F1 PASS" if abs(o[0] - t[0]) <= 0.05 and abs(o[1] - t[1]) <= 0.05 else "F1 OUTSIDE +-0.05",
          f"| worst cell |hF1 diff| {max(abs(a[0] - b[0]) for a, b in zip(ours_all, t1_all)):.2f}")
    pr = paper_raw_verdicts(); ours_v = [None if x is None else x > .5 for x in sc]
    both = [(a, b) for a, b in zip(ours_v, pr) if a is not None and b is not None]
    # Primary check: the paper's own two independent runs agree at kappa 0.69 (GPT-5-mini) .. 0.56 (Mistral Large).
    print(f"agreement with the paper's shipped verdicts: {np.mean([a == b for a, b in both]):.2f} on {len(both)} samples,"
          f" kappa {cohen_kappa_score([a for a, _ in both], [b for _, b in both]):.2f}")
    ok = [(s, x) for s, x in zip(S, sc) if x is not None]
    print(f"AUROC (LLM-positive, all 7 generators, valid only): {roc_auc_score([not s['human'] for s, _ in ok], [x for _, x in ok]):.3f}")

In [ ]:
# %% 5. detectors/llm_judge.py: standalone module (copy this cell as the file); smoke-tested on 3 samples
import functools, os, re, json, concurrent.futures as cf
from openai import OpenAI

JUDGE = os.environ.get("LLM_JUDGE_MODEL", JUDGE)   # in the repo: os.environ["LLM_JUDGE_MODEL"], set from config.toml
PROMPT = PROMPTS["classification"]                  # in the repo: paste Djire's zero-shot prompt string verbatim
N_VOTES, TEMPERATURE, MAX_TOKENS, RETRIES = 3, 0.1, 8192, 3

@functools.cache
def _client():
    return OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

def _verdict(text):  # "yes" | "no" | None; JSON first, then the regex Table 1 needs for Mistral replies
    text = (text or "").strip()
    for cand in (text, text[text.find("{"):text.rfind("}") + 1]):
        try:
            v = str(json.loads(cand).get("response", "")).strip().lower()
            if v in ("yes", "no"):
                return v
        except Exception:
            pass
    m = re.search(r'"response"\s*:\s*"\s*(yes|no)\b', text, re.I)
    return m.group(1).lower() if m else None

def _one(code):
    votes = []
    for _ in range(N_VOTES):
        for _ in range(1 + RETRIES):
            r = _client().chat.completions.create(model=JUDGE, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                messages=[{"role": "system", "content": PROMPT}, {"role": "user", "content": code}])
            v = _verdict(r.choices[0].message.content)
            if v:
                votes.append(v); break
    if not votes:
        raise RuntimeError("llm_judge: no valid verdict after retries; rerun the score stage")
    return sum(v == "yes" for v in votes) / len(votes)   # Score in {0, 1/3, 2/3, 1}; threshold 0.5 = majority

def score(codes):
    with cf.ThreadPoolExecutor(8) as ex:
        return list(ex.map(_one, codes))

smoke = [S[0], S[18], S[40]]   # human, ChatGPT, other AI
print("\nscore() smoke test (threshold 0.5 = majority of votes):")
for s, p in zip(smoke, score([s["code"] for s in smoke])):
    print(f"score {p:.4f} -> predict {'AI' if p >= 0.5 else 'Human'}, truth {'Human' if s['human'] else 'AI (' + s['gen'] + ')'}")
    print("\n".join(s["code"].splitlines()[:5]), "\n" + "-" * 40)